## Training workflow

This notebook is developed and executed in Google Colab to make use of GPU acceleration.

In [2]:
# GitHub setup

from google.colab import userdata
import os

# Load GitHub token from Colab Secrets
os.environ["GITHUB_TOKEN"] = userdata.get("NLP-II-PolitiSky24Token")

# Configure temporary GitHub authentication
!git config --global credential.helper cache
!printf "protocol=https\nhost=github.com\nusername=jakobtitz\npassword=$GITHUB_TOKEN\n\n" | git credential approve

# Clone the RoBERTa training branch
!git clone --branch JT_creating_roberta_evaluations https://github.com/jakobtitz/nlp-II-politiksky24.git

# Move into repository
%cd /content/nlp-II-politiksky24

# Verify setup
!git branch --show-current
!git status

Cloning into 'nlp-II-politiksky24'...
remote: Enumerating objects: 252, done.
remote: Counting objects: 100% (252/252), done.
remote: Compressing objects: 100% (207/207), done.
remote: Total 252 (delta 81), reused 193 (delta 38), pack-reused 0 (from 0)
Receiving objects: 100% (252/252), 8.44 MiB | 11.61 MiB/s, done.
Resolving deltas: 100% (81/81), done.
Filtering content: 100% (31/31), 314.94 MiB | 9.96 MiB/s, done.
/content/nlp-II-politiksky24
JT_creating_roberta_evaluations
On branch JT_creating_roberta_evaluations
Your branch is up to date with 'origin/JT_creating_roberta_evaluations'.

nothing to commit, working tree clean


---
## **Evaluate Input Interventions**

This notebook evaluates the controlled input interventions created in `04.1_create_input_interventions.ipynb` using the trained RoBERTa classifier.

The objective is to measure how model predictions change when specific information sources are systematically modified:

1. the supplied target identity,
2. explicit candidate mentions in the retrieved context posts, and
3. label-correlated lexical cues.

All intervention datasets were constructed from the held-out human-annotated test set. Any data-driven intervention rules were derived exclusively from the training data before being applied to the test set.

The RoBERTa model is loaded in its previously trained and frozen state.

Predictions on each modified input are compared with predictions on the corresponding original test example. For label-preserving interventions, changes in classification performance are also evaluated against the human gold labels. Target swapping is treated separately because the original stance label is no longer a valid gold label after changing the target.

---

### **1. Setup**

Load the libraries and define the paths to the fixed human test set, the frozen RoBERTa model, and the previously generated intervention datasets.

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

from sklearn.metrics import classification_report, f1_score


DATA_DIR = Path("data/preprocessed")
INTERVENTION_DIR = Path("data/interventions")

HUMAN_TEST_PATH = DATA_DIR / "human_test.parquet"

MODEL_DIR = Path(
    "/content/drive/MyDrive/NLP_II/models/roberta_base_best"
)

RESULTS_DIR = Path(
    "/content/drive/MyDrive/NLP_II/results/roberta_interventions"
)

In [4]:
# Collect paths for all intervention variants.
intervention_paths = {
    "target_masked": (
        INTERVENTION_DIR / "human_test_target_masked.parquet"
    ),
    "target_swapped": (
        INTERVENTION_DIR / "human_test_target_swapped.parquet"
    ),
    "candidate_mentions_masked": (
        INTERVENTION_DIR / "human_test_candidate_mentions_masked.parquet"
    ),
    "lexical_cues_top10_masked": (
        INTERVENTION_DIR / "human_test_lexical_cues_top10_masked.parquet"
    ),
    "lexical_cues_top25_masked": (
        INTERVENTION_DIR / "human_test_lexical_cues_top25_masked.parquet"
    )
}

for seed in range(1, 6):
    intervention_paths[
        f"candidate_mentions_control_seed{seed}"
    ] = (
        INTERVENTION_DIR
        / f"human_test_candidate_mentions_control_seed{seed}.parquet"
    )

    intervention_paths[
        f"lexical_cues_top10_control_seed{seed}"
    ] = (
        INTERVENTION_DIR
        / f"human_test_lexical_cues_top10_control_seed{seed}.parquet"
    )

    intervention_paths[
        f"lexical_cues_top25_control_seed{seed}"
    ] = (
        INTERVENTION_DIR
        / f"human_test_lexical_cues_top25_control_seed{seed}.parquet"
    )

---

### **Load the fixed test set and frozen model**

The original human-annotated test set provides the reference examples and gold labels. The previously saved RoBERTa model and tokenizer are loaded without modification and will be used for all original and intervention inputs.

In [5]:
from google.colab import drive

drive.mount("/content/drive")

human_test = pd.read_parquet(HUMAN_TEST_PATH)

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

roberta = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

roberta.to(device)
roberta.eval()

print(f"Human test examples: {len(human_test):,}")
print(f"Columns: {human_test.columns.tolist()}")
print(f"Device: {device}")

Mounted at /content/drive


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Human test examples: 890
Columns: ['UserId', 'TargetEntity', 'StanceLabel', 'ContextPosts']
Device: cuda


---

### **2. Inspect the Frozen RoBERTa Model**

Before applying the model to the test interventions, inspect the loaded model and tokenizer and verify the label mapping used during training.

The intervention placeholder is also inspected under the frozen tokenizer to document how the synthetic masking token is represented by RoBERTa.

In [6]:
print(f"Model type: {roberta.config.model_type}")
print(f"Number of labels: {roberta.config.num_labels}")
print(f"Maximum tokenizer length: {tokenizer.model_max_length}")
print(f"Separator token: {tokenizer.sep_token}")
print(f"Padding token: {tokenizer.pad_token}")
print(f"Model label mapping: {roberta.config.label2id}")

# Define the label mapping used throughout the evaluation
LABEL2ID = {
    "Against": 0,
    "Favor": 1,
    "Neither": 2,
}

ID2LABEL = {
    value: key
    for key, value in LABEL2ID.items()
}

LABEL_ORDER = ["Against", "Favor", "Neither"]

# Verify that the model's internal label order matches the expected label mapping
assert roberta.config.label2id == LABEL2ID
assert roberta.config.id2label == ID2LABEL

print(
    "Model label order:",
    [
        ID2LABEL[class_id]
        for class_id in range(roberta.config.num_labels)
    ],
)

Model type: roberta
Number of labels: 3
Maximum tokenizer length: 512
Separator token: </s>
Padding token: <pad>
Model label mapping: {'Against': 0, 'Favor': 1, 'Neither': 2}
Model label order: ['Against', 'Favor', 'Neither']


---

### **Verify the frozen context-masking token**

All context-level masking interventions were created with the synthetic token `requ`, which was selected and frozen before intervention generation in `04_select_masking_token.ipynb`.

For RoBERTa, the token is inspected under the frozen tokenizer to document how it is represented by the model. Target masking is treated separately and uses an empty target string rather than a synthetic token.

In [7]:
CONTEXT_MASK_TOKEN = "requ"

mask_tokens = tokenizer.tokenize(CONTEXT_MASK_TOKEN)
mask_token_ids = tokenizer.encode(
    CONTEXT_MASK_TOKEN,
    add_special_tokens=False,
)

print("RoBERTa tokens:", mask_tokens)
print("Token IDs:", mask_token_ids)

RoBERTa tokens: ['requ']
Token IDs: [42172]


---

### **3. Load Intervention Datasets**

Load all previously generated intervention datasets. Each dataset contains the same held-out test examples in the same row order as the original test set, allowing predictions to be compared pairwise with the corresponding original input.

In [8]:
intervention_data = {
    name: pd.read_parquet(path)
    for name, path in intervention_paths.items()
}

print(f"Loaded {len(intervention_data)} intervention datasets.")

for name in intervention_data:
    print(name)

Loaded 20 intervention datasets.
target_masked
target_swapped
candidate_mentions_masked
lexical_cues_top10_masked
lexical_cues_top25_masked
candidate_mentions_control_seed1
lexical_cues_top10_control_seed1
lexical_cues_top25_control_seed1
candidate_mentions_control_seed2
lexical_cues_top10_control_seed2
lexical_cues_top25_control_seed2
candidate_mentions_control_seed3
lexical_cues_top10_control_seed3
lexical_cues_top25_control_seed3
candidate_mentions_control_seed4
lexical_cues_top10_control_seed4
lexical_cues_top25_control_seed4
candidate_mentions_control_seed5
lexical_cues_top10_control_seed5
lexical_cues_top25_control_seed5


---

### **4. Construct Model Inputs**

Reconstruct the model inputs for the original test set and all intervention datasets using exactly the same input-construction function as during RoBERTa training.

Keeping the input construction unchanged ensures that any prediction differences are caused by the interventions rather than by differences in preprocessing or formatting.

In [9]:
# Combine all valid context posts into the same context representation used during training
def build_context_text(context_posts):
    posts = [
        post["Content"]
        for post in context_posts
        if post["Content"] is not None
    ]

    return f" {tokenizer.sep_token} ".join(posts)


# Apply the same input-construction function to the original and all intervention variants
human_test["ContextText"] = (
    human_test["ContextPosts"].apply(build_context_text)
)

for name, df in intervention_data.items():
    df["ContextText"] = (
        df["ContextPosts"].apply(build_context_text)
    )

---

### **5. Generate Original Test Predictions**

Generate predictions and class probabilities for the unchanged human-annotated test set.

These outputs serve as the reference for all subsequent intervention comparisons. The frozen model is applied to the human-annotated test set without any refitting or adaptation.

In [10]:
# Generate predictions and class probabilities in batches
def predict_roberta(df, batch_size=16):
    all_pred_ids = []
    all_probabilities = []

    targets = df["TargetEntity"].tolist()
    contexts = df["ContextText"].tolist()

    roberta.eval()

    with torch.inference_mode():
        for start in range(0, len(df), batch_size):
            end = start + batch_size

            batch_targets = targets[start:end]
            batch_contexts = contexts[start:end]

            encoded = tokenizer(
                batch_targets,
                batch_contexts,
                max_length=512,
                truncation="only_second",
                padding=True,
                return_tensors="pt",
            )

            encoded = {
                key: value.to(device)
                for key, value in encoded.items()
            }

            outputs = roberta(**encoded)

            probabilities = torch.softmax(
                outputs.logits,
                dim=-1,
            )

            pred_ids = torch.argmax(
                probabilities,
                dim=-1,
            )

            all_pred_ids.extend(
                pred_ids.cpu().numpy()
            )

            all_probabilities.append(
                probabilities.cpu().numpy()
            )

    return (
        np.array(all_pred_ids),
        np.concatenate(all_probabilities, axis=0),
    )

In [11]:
original_pred_ids, original_probabilities = predict_roberta(
    human_test
)

original_predictions = np.array([
    ID2LABEL[pred_id]
    for pred_id in original_pred_ids
])

print(f"Predictions: {len(original_predictions):,}")
print(f"Probability matrix: {original_probabilities.shape}")

Predictions: 890
Probability matrix: (890, 3)


---

### **6. Evaluate Original Test Performance**

Evaluate the unchanged human-annotated test set to establish the RoBERTa reference performance.

Macro-F1 is used as the main overall metric because the stance classes are imbalanced. Class-specific precision, recall, and F1-scores are additionally reported to inspect performance differences between stance classes.

In [12]:
y_true = human_test["StanceLabel"].to_numpy()

original_macro_f1 = f1_score(
    y_true,
    original_predictions,
    labels=LABEL_ORDER,
    average="macro",
)

print(f"Original Macro-F1: {original_macro_f1:.4f}")

Original Macro-F1: 0.8123


In [13]:
# Compute the reference classification report for the unmodified test set
original_report = classification_report(
    y_true,
    original_predictions,
    labels=LABEL_ORDER,
    output_dict=True,
    zero_division=0,
)

original_report = pd.DataFrame(original_report).T

original_report

,precision,recall,f1-score,support
Against,0.911308,0.934091,0.922559,440.000000
Favor,0.800885,0.770213,0.785249,235.000000
Neither,0.732394,0.725581,0.728972,215.000000
accuracy,0.840449,0.840449,0.840449,0.840449
macro avg,0.814863,0.809962,0.812260,890.000000
weighted avg,0.838931,0.840449,0.839538,890.000000


The frozen RoBERTa model achieves a Macro-F1 of 0.812 and an accuracy of 0.840 on the unmodified human-annotated test set. Performance is strongest for Against (F1 = 0.923), followed by Favor (0.785), while Neither is the most difficult class (0.729).

---

### **7. Generate Intervention Predictions**

Apply the same frozen RoBERTa model to every intervention dataset.

For each intervention, store both the predicted stance labels and the corresponding class probabilities. These outputs will later be compared pairwise with the predictions on the unchanged test inputs.

In [14]:
intervention_predictions = {}
intervention_probabilities = {}

# Apply the frozen RoBERTa model to all intervention variants
for name, df in intervention_data.items():

    pred_ids, probabilities = predict_roberta(df)

    predictions = np.array([
        ID2LABEL[pred_id]
        for pred_id in pred_ids
    ])

    intervention_predictions[name] = predictions
    intervention_probabilities[name] = probabilities

---

### **8. Compare Intervention Outcomes**

Compare each intervention with the predictions on the unchanged test inputs.

For label-preserving interventions, Macro-F1 is calculated against the human gold labels and its change relative to the original performance is reported. The prediction flip rate measures the proportion of examples for which the predicted stance label changes after the intervention.

Target swapping is evaluated only through prediction changes because changing the target invalidates the original gold stance label.

In [15]:
# Compare each intervention with the original predictions using Macro-F1 change and prediction flip rate
comparison_rows = [
    {
        "condition": "original",
        "macro_f1": original_macro_f1,
        "delta_macro_f1": 0.0,
        "flip_rate": 0.0,
    }
]

for name, predictions in intervention_predictions.items():

    flip_rate = np.mean(
        predictions != original_predictions
    )

    # Target swapping has no valid gold labels
    if name == "target_swapped":
        macro_f1 = np.nan
        delta_macro_f1 = np.nan

    else:
        macro_f1 = f1_score(
            y_true,
            predictions,
            labels=LABEL_ORDER,
            average="macro",
        )

        delta_macro_f1 = (
            macro_f1 - original_macro_f1
        )

    comparison_rows.append(
        {
            "condition": name,
            "macro_f1": macro_f1,
            "delta_macro_f1": delta_macro_f1,
            "flip_rate": flip_rate,
        }
    )

comparison_results = pd.DataFrame(
    comparison_rows
)

comparison_results

,condition,macro_f1,delta_macro_f1,flip_rate
0,original,0.812260,0.000000,0.000000
1,target_masked,0.742188,-0.070072,0.113483
2,target_swapped,NaN,NaN,0.715730
3,candidate_mentions_masked,0.554192,-0.258068,0.389888
4,lexical_cues_top10_masked,0.814043,0.001783,0.004494
5,lexical_cues_top25_masked,0.812818,0.000558,0.005618
6,candidate_mentions_control_seed1,0.804579,-0.007682,0.037079
7,lexical_cues_top10_control_seed1,0.813232,0.000972,0.001124
8,lexical_cues_top25_control_seed1,0.810291,-0.001969,0.005618
9,candidate_mentions_control_seed2,0.804225,-0.008035,0.042697


The RoBERTa classifier shows substantial sensitivity to both the supplied target and explicit candidate mentions in the retrieved context. Target masking reduces Macro-F1 by approximately 0.070 and changes 11.3% of predictions, while target swapping changes 71.6% of predictions. Candidate-mention masking has an even stronger effect, reducing Macro-F1 by approximately 0.258 and changing 39.0% of predictions. In contrast, masking the identified top-10 or top-25 lexical cues produces almost no change in performance or predicted labels.

---

### **9. Compare Masking Interventions with Matched Controls**

Aggregate the five matched-control runs for each masking intervention.

The control conditions replace approximately the same number of tokens as the corresponding masking intervention, but at matched non-target locations. Comparing the actual intervention with the average control effect helps distinguish reliance on the selected information from the general effect of modifying the same amount of input text.

In [16]:
control_groups = {
    "candidate_mentions_masked": "candidate_mentions_control_",
    "lexical_cues_top10_masked": "lexical_cues_top10_control_",
    "lexical_cues_top25_masked": "lexical_cues_top25_control_",
}

control_comparison_rows = []

# Compare each masking intervention with its five matched controls
for intervention, control_prefix in control_groups.items():

    intervention_row = comparison_results.loc[
        comparison_results["condition"] == intervention
    ].iloc[0]

    controls = comparison_results[
        comparison_results["condition"].str.startswith(
            control_prefix
        )
    ]

    assert len(controls) == 5

    control_comparison_rows.append(
        {
            "intervention": intervention,
            "intervention_macro_f1": intervention_row["macro_f1"],
            "control_macro_f1_mean": controls["macro_f1"].mean(),
            "control_macro_f1_sd": controls["macro_f1"].std(),
            "macro_f1_vs_control": (
                intervention_row["macro_f1"]
                - controls["macro_f1"].mean()
            ),
            "intervention_flip_rate": intervention_row["flip_rate"],
            "control_flip_rate_mean": controls["flip_rate"].mean(),
            "control_flip_rate_sd": controls["flip_rate"].std(),
            "flip_rate_vs_control": (
                intervention_row["flip_rate"]
                - controls["flip_rate"].mean()
            ),
        }
    )

control_comparison = pd.DataFrame(
    control_comparison_rows
)

control_comparison

,intervention,intervention_macro_f1,control_macro_f1_mean,control_macro_f1_sd,macro_f1_vs_control,intervention_flip_rate,control_flip_rate_mean,control_flip_rate_sd,flip_rate_vs_control
0,candidate_mentions_masked,0.554192,0.807343,0.005085,-0.253151,0.389888,0.035955,0.004424,0.353933
1,lexical_cues_top10_masked,0.814043,0.812832,0.000420,0.001212,0.004494,0.001798,0.000615,0.002697
2,lexical_cues_top25_masked,0.812818,0.811984,0.001388,0.000834,0.005618,0.004270,0.001846,0.001348


Masking explicit candidate mentions produces a substantially stronger effect than matched random removals. Macro-F1 decreases to 0.554 compared with an average of 0.807 under the controls, while the prediction flip rate increases from approximately 3.6% to 39.0%. In contrast, masking the top-10 and top-25 label-associated lexical cues produces performance and prediction changes that remain very close to their matched controls.

---

### **10. Analyze Changes in Predicted Probabilities**

Prediction flips capture only cases in which an intervention changes the final predicted stance label. Smaller changes in model confidence can occur even when the predicted label remains unchanged.

For each intervention, we therefore measure:

- the mean change in probability assigned to the originally predicted class, and
- the mean absolute change across all three class probabilities.

Negative changes in the original-class probability indicate that the intervention weakens the model's original decision.

In [17]:
original_pred_indices = np.array([
    LABEL2ID[label]
    for label in original_predictions
])

row_indices = np.arange(len(human_test))

original_predicted_class_prob = original_probabilities[
    row_indices,
    original_pred_indices,
]

probability_rows = []

# Measure how each intervention changes the predicted probabilities relative to the original inputs
for name, probabilities in intervention_probabilities.items():

    intervention_original_class_prob = probabilities[
        row_indices,
        original_pred_indices,
    ]

    original_class_probability_change = (
        intervention_original_class_prob
        - original_predicted_class_prob
    )

    mean_absolute_probability_change = np.abs(
        probabilities - original_probabilities
    ).mean()

    probability_rows.append(
        {
            "condition": name,
            "mean_original_class_probability_change":
                original_class_probability_change.mean(),
            "mean_absolute_probability_change":
                mean_absolute_probability_change,
        }
    )

probability_results = pd.DataFrame(probability_rows)

probability_results

,condition,mean_original_class_probability_change,mean_absolute_probability_change
0,target_masked,-0.078217,0.072664
1,target_swapped,-0.622788,0.443337
2,candidate_mentions_masked,-0.336434,0.258036
3,lexical_cues_top10_masked,-0.001925,0.001942
4,lexical_cues_top25_masked,-0.001395,0.003780
5,candidate_mentions_control_seed1,-0.017865,0.021953
6,lexical_cues_top10_control_seed1,-0.000056,0.001017
7,lexical_cues_top25_control_seed1,-0.001826,0.002682
8,candidate_mentions_control_seed2,-0.019963,0.022105
9,lexical_cues_top10_control_seed2,-0.000338,0.000943


In [18]:
probability_control_rows = []

# Compare probability changes from each masking intervention with its matched controls
for intervention, control_prefix in control_groups.items():

    intervention_row = probability_results.loc[
        probability_results["condition"] == intervention
    ].iloc[0]

    controls = probability_results[
        probability_results["condition"].str.startswith(
            control_prefix
        )
    ]

    assert len(controls) == 5

    probability_control_rows.append(
        {
            "intervention": intervention,

            "intervention_original_class_prob_change":
                intervention_row[
                    "mean_original_class_probability_change"
                ],

            "control_original_class_prob_change_mean":
                controls[
                    "mean_original_class_probability_change"
                ].mean(),

            "control_original_class_prob_change_sd":
                controls[
                    "mean_original_class_probability_change"
                ].std(),

            "intervention_mean_abs_prob_change":
                intervention_row[
                    "mean_absolute_probability_change"
                ],

            "control_mean_abs_prob_change_mean":
                controls[
                    "mean_absolute_probability_change"
                ].mean(),

            "control_mean_abs_prob_change_sd":
                controls[
                    "mean_absolute_probability_change"
                ].std(),
        }
    )

probability_control_comparison = pd.DataFrame(
    probability_control_rows
)

probability_control_comparison

,intervention,intervention_original_class_prob_change,control_original_class_prob_change_mean,control_original_class_prob_change_sd,intervention_mean_abs_prob_change,control_mean_abs_prob_change_mean,control_mean_abs_prob_change_sd
0,candidate_mentions_masked,-0.336434,-0.018844,0.001095,0.258036,0.022342,0.000548
1,lexical_cues_top10_masked,-0.001925,-0.000753,0.000625,0.001942,0.001270,0.000354
2,lexical_cues_top25_masked,-0.001395,-0.001875,0.000388,0.003780,0.002727,0.000381


Probability-based results reinforce the performance-based findings. Target swapping produces the strongest change in RoBERTa's predicted probability distributions, while masking explicit candidate mentions also substantially reduces the probability assigned to the model's original prediction. Matched candidate-control removals produce only minor changes. In contrast, masking the top-10 and top-25 lexical cue sets results in only small probability shifts that remain close to the corresponding controls.

---

### **11. Analyze Intervention Effects by Target**

Evaluate whether intervention effects differ between the two political targets.

All examples are grouped by their original target in the human-annotated test set. For label-preserving interventions, target-specific Macro-F1 and prediction flip rates are reported. For target swapping, only behavioral changes are evaluated because the original gold label is no longer valid after changing the target.

The preceding dataset analysis showed a strong association between target and stance label in both the training and human-annotated test data. Target-specific Macro-F1 is therefore retained as a performance measure, but absolute Macro-F1 values are not interpreted as directly comparable measures of difficulty across targets. The main focus is on within-target changes relative to the original predictions.

In [19]:
target_rows = []

# Recompute intervention effects separately for each target to assess whether model sensitivity differs between Trump and Harris
for target in human_test["TargetEntity"].unique():

    target_mask = (
        human_test["TargetEntity"].to_numpy() == target
    )

    target_indices = np.where(target_mask)[0]

    target_y_true = y_true[target_mask]
    target_original_predictions = original_predictions[target_mask]

    original_target_macro_f1 = f1_score(
        target_y_true,
        target_original_predictions,
        labels=LABEL_ORDER,
        average="macro",
    )

    original_target_class_prob = original_probabilities[
        target_indices,
        original_pred_indices[target_indices],
    ]

    target_rows.append(
        {
            "target": target,
            "condition": "original",
            "macro_f1": original_target_macro_f1,
            "delta_macro_f1": 0.0,
            "flip_rate": 0.0,
            "mean_original_class_probability_change": 0.0,
            "mean_absolute_probability_change": 0.0,
        }
    )

    for name, predictions in intervention_predictions.items():

        target_predictions = predictions[target_mask]

        target_probabilities = intervention_probabilities[name][
            target_indices
        ]

        flip_rate = np.mean(
            target_predictions
            != target_original_predictions
        )

        intervention_target_class_prob = target_probabilities[
            np.arange(len(target_indices)),
            original_pred_indices[target_indices],
        ]

        mean_original_class_probability_change = (
            intervention_target_class_prob
            - original_target_class_prob
        ).mean()

        mean_absolute_probability_change = np.abs(
            target_probabilities
            - original_probabilities[target_indices]
        ).mean()

        if name == "target_swapped":
            macro_f1 = np.nan
            delta_macro_f1 = np.nan

        else:
            macro_f1 = f1_score(
                target_y_true,
                target_predictions,
                labels=LABEL_ORDER,
                average="macro",
            )

            delta_macro_f1 = (
                macro_f1
                - original_target_macro_f1
            )

        target_rows.append(
            {
                "target": target,
                "condition": name,
                "macro_f1": macro_f1,
                "delta_macro_f1": delta_macro_f1,
                "flip_rate": flip_rate,
                "mean_original_class_probability_change":
                    mean_original_class_probability_change,
                "mean_absolute_probability_change":
                    mean_absolute_probability_change,
            }
        )

target_results = pd.DataFrame(target_rows)

In [20]:
main_conditions = [
    "original",
    "target_masked",
    "target_swapped",
    "candidate_mentions_masked",
    "lexical_cues_top10_masked",
    "lexical_cues_top25_masked",
]

target_results[
    target_results["condition"].isin(main_conditions)
].reset_index(drop=True)

,target,condition,macro_f1,delta_macro_f1,flip_rate,mean_original_class_probability_change,mean_absolute_probability_change
0,Trump,original,0.688770,0.000000,0.000000,0.000000,0.000000
1,Trump,target_masked,0.654556,-0.034214,0.092135,-0.070757,0.057297
2,Trump,target_swapped,NaN,NaN,0.914607,-0.816437,0.557189
3,Trump,candidate_mentions_masked,0.601696,-0.087074,0.357303,-0.353563,0.247645
4,Trump,lexical_cues_top10_masked,0.693766,0.004996,0.002247,-0.002035,0.001937
5,Trump,lexical_cues_top25_masked,0.693766,0.004996,0.002247,-0.000244,0.004034
6,Harris,original,0.728334,0.000000,0.000000,0.000000,0.000000
7,Harris,target_masked,0.622490,-0.105844,0.134831,-0.085678,0.088031
8,Harris,target_swapped,NaN,NaN,0.516854,-0.429139,0.329486
9,Harris,candidate_mentions_masked,0.429207,-0.299128,0.422472,-0.319305,0.268428
